<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.1-document-ai/notebooks/GCP_Capstone_4.1_Document_AI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.1 Document AI — OCR, Layout Parser, Form Parser
**Netsetos GenAI Engineering — GCP Capstone**

Convert PDFs, scans, and forms into clean text for RAG.


## Setup


In [ ]:
!pip install -q google-cloud-documentai google-cloud-documentai-toolbox
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'us'
# Create processors in Console, paste IDs below:
OCR_ID = 'YOUR_OCR_PROCESSOR_ID'
LAYOUT_ID = 'YOUR_LAYOUT_PROCESSOR_ID'
FORM_ID = 'YOUR_FORM_PROCESSOR_ID'


## Cell 1: Universal Processing Function


In [ ]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

def process_document(project_id, location, processor_id, file_path,
                     mime_type='application/pdf', process_options=None):
    client = documentai.DocumentProcessorServiceClient(
        client_options=ClientOptions(
            api_endpoint=f'{location}-documentai.googleapis.com'))
    name = client.processor_path(project_id, location, processor_id)
    with open(file_path, 'rb') as f:
        content = f.read()
    request = documentai.ProcessRequest(
        name=name,
        raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
        process_options=process_options)
    return client.process_document(request=request).document

print('process_document() ready')


## Cell 2: OCR — Full Text Extraction


In [ ]:
# Upload a test PDF to Colab first
# from google.colab import files
# uploaded = files.upload()

options = documentai.ProcessOptions(
    ocr_config=documentai.OcrConfig(
        enable_native_pdf_parsing=True,
        enable_image_quality_scores=True,
        hints=documentai.OcrConfig.Hints(language_hints=['en','hi'])))

doc = process_document(PROJECT_ID, LOCATION, OCR_ID, 'test.pdf',
                       process_options=options)

print(f'Full text ({len(doc.text)} chars):')
print(doc.text[:500])
print(f'\nPages: {len(doc.pages)}')
for page in doc.pages:
    langs = [(l.language_code, f'{l.confidence:.0%}') for l in page.detected_languages]
    print(f'  Page {page.page_number}: {len(page.paragraphs)} paragraphs, langs={langs}')


## Cell 3: Layout Parser — RAG-Ready Chunks


In [ ]:
options = documentai.ProcessOptions(
    layout_config=documentai.ProcessOptions.LayoutConfig(
        enable_table_annotation=True,
        enable_image_annotation=True,
        chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
            chunk_size=1024,
            include_ancestor_headings=True)))

doc = process_document(PROJECT_ID, LOCATION, LAYOUT_ID, 'test.pdf',
                       process_options=options)

print(f'Chunks: {len(doc.chunked_document.chunks)}')
for i, chunk in enumerate(doc.chunked_document.chunks[:5]):
    print(f'\n--- Chunk {i} ({chunk.chunk_id}) ---')
    print(f'Pages: {chunk.page_span.page_start}-{chunk.page_span.page_end}')
    print(f'Content:\n{chunk.content[:200]}...')


## Cell 4: Form Parser — Key-Value Pairs


In [ ]:
doc = process_document(PROJECT_ID, LOCATION, FORM_ID, 'form.pdf')

for page in doc.pages:
    print(f'\n=== Page {page.page_number} ===')
    for field in page.form_fields:
        key = field.field_name.text_anchor.content.strip()
        val = field.field_value.text_anchor.content.strip()
        print(f'  {key}: {val} ({field.field_value.confidence:.0%})')
    for idx, table in enumerate(page.tables):
        print(f'\n  Table {idx}:')
        for row in table.body_rows:
            cells = [c.layout.text_anchor.content.strip() for c in row.cells]
            print(f'    {cells}')


## Cell 5: Store Chunks in Firestore


In [ ]:
from google.cloud import firestore

def store_chunks(document, source_file):
    db = firestore.Client()
    batch = db.batch()
    for chunk in document.chunked_document.chunks:
        ref = db.collection('rag_chunks').document(chunk.chunk_id)
        batch.set(ref, {
            'source_file': source_file,
            'content': chunk.content,
            'page_start': chunk.page_span.page_start,
            'page_end': chunk.page_span.page_end,
            'processed_at': firestore.SERVER_TIMESTAMP})
    batch.commit()
    print(f'Stored {len(document.chunked_document.chunks)} chunks')

# store_chunks(doc, 'test.pdf')  # Uncomment to store


## Cell 6: Document Ingestion Module


In [ ]:
class DocumentIngester:
    def __init__(self, project_id, location='us'):
        self.project_id = project_id
        self.location = location
        self.client = documentai.DocumentProcessorServiceClient(
            client_options=ClientOptions(
                api_endpoint=f'{location}-documentai.googleapis.com'))
        self.db = firestore.Client()

    def process(self, processor_id, file_path, mime_type='application/pdf',
                 process_options=None):
        name = self.client.processor_path(self.project_id, self.location, processor_id)
        with open(file_path, 'rb') as f:
            content = f.read()
        request = documentai.ProcessRequest(
            name=name,
            raw_document=documentai.RawDocument(content=content, mime_type=mime_type),
            process_options=process_options)
        return self.client.process_document(request=request).document

    def ingest_for_rag(self, layout_id, file_path, chunk_size=1024):
        options = documentai.ProcessOptions(
            layout_config=documentai.ProcessOptions.LayoutConfig(
                chunking_config=documentai.ProcessOptions.LayoutConfig.ChunkingConfig(
                    chunk_size=chunk_size,
                    include_ancestor_headings=True)))
        doc = self.process(layout_id, file_path, process_options=options)
        batch = self.db.batch()
        for chunk in doc.chunked_document.chunks:
            ref = self.db.collection('rag_chunks').document(chunk.chunk_id)
            batch.set(ref, {
                'content': chunk.content,
                'source': file_path,
                'processed_at': firestore.SERVER_TIMESTAMP})
        batch.commit()
        return len(doc.chunked_document.chunks)

print('DocumentIngester ready')


## ✅ Lesson 4.1 Complete!

- ✅ Universal processing pattern (ProcessRequest + RawDocument)
- ✅ Enterprise OCR with quality scores and Hindi support
- ✅ Layout Parser with include_ancestor_headings for RAG
- ✅ Form Parser with key-value pairs and tables
- ✅ Batch processing for large documents
- ✅ Firestore storage for chunks
- ✅ DocumentIngester production module

**Next: Lesson 4.2 — Naive RAG Pipeline**
